In [ ]:
%apt install libdb5.3-dev
%pip install gutenberg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  db5.3-doc
The following NEW packages will be installed:
  libdb5.3-dev
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 830 kB of archives.
After this operation, 3,151 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libdb5.3-dev amd64 5.3.28+dfsg1-0.8ubuntu3 [830 kB]
Fetched 830 kB in 2s (394 kB/s)
Selecting previously unselected package libdb5.3-dev.
(Reading database ... 126209 files and directories currently installed.)
Preparing to unpack .../libdb5.3-dev_5.3.28+dfsg1-0.8ubuntu3_amd64.deb ...
Unpacking libdb5.3-dev (5.3.28+dfsg1-0.8ubuntu3) ...
Setting up libdb5.3-dev (5.3.28+dfsg1-0.8ubuntu3) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of sparqlwrapper to determine 

In [1]:
import os
import pandas as pd
import requests
import numpy as np
from bs4 import BeautifulSoup
from urllib.request import urlopen
from gutenberg.acquire import load_etext
from gutenberg.cleanup import strip_headers
import pickle

In [2]:
# only removes funny tokens for English texts
def remove_funny_tokens(text):
    tokens = text.split()
    sample = ' '.join(' '.join(tokens).replace('xe2x80x9c', ' ').replace('xe2x80x9d', ' ')\
                                      .replace('xe2x80x94', ' ').replace('xe2x80x99', "'")\
                                      .replace('xe2x80x98', "'").split())
    return sample

# clean newlines, carriage returns and tabs
def clean_text(text):
    cleaned_listed_text = []
    listed_text = list(text)

    for iter in range(len(listed_text) - 1):
        if (listed_text[iter] == '\\' and listed_text[iter + 1] == 'n') or \
            (listed_text[iter] == 'n' and listed_text[iter - 1] == '\\'):
            continue
        elif listed_text[iter] == '\\' and listed_text[iter + 1] == 'r' or \
            (listed_text[iter] == 'r' and listed_text[iter - 1] == '\\'):
            continue
        elif listed_text[iter] == '\\' and listed_text[iter + 1] == 't' or \
            (listed_text[iter] == 't' and listed_text[iter - 1] == '\\'):
            continue
        elif listed_text[iter] == '\\':
            continue
        else:
            cleaned_listed_text.append(listed_text[iter])

    cleaned_text = ''.join([str(char) for char in cleaned_listed_text])
    cleaned_text = remove_funny_tokens(cleaned_text)

    return ''.join(cleaned_text)

In [3]:
df_metadata = pd.read_csv('gutenberg_metadata.csv')
sample_idx = np.random.choice(df_metadata.index, 1000)
sample_df = df_metadata.loc[sample_idx].copy()

In [ ]:
data = {'Author': None, 'Title': None, 'Link': None, 'ID': None, 'Bookshelf': None, 'Text': None}

for key, row in sample_df.iterrows():
    if data['Author'] == None:
        data['Author'] = [row['Author']]
    else:
        data['Author'].append(row['Author'])

    if data['Title'] == None:
        data['Title'] = [row['Title']]
    else:
        data['Title'].append(row['Title'])

    if data['Link'] == None:
        data['Link'] = [row['Link']]
    else:
        data['Link'].append(row['Link'])

    book_id = int(row['Link'].split('/')[-1])

    if data['ID'] == None:
        data['ID'] = [book_id]
    else:
        data['ID'].append(book_id)

    if data['Bookshelf'] == None:
        data['Bookshelf'] = [row['Bookshelf']]
    else:
        data['Bookshelf'].append(row['Bookshelf'])

    text = np.nan
    try:
        text = strip_headers(load_etext(etextno=book_id,
                                        mirror='http://www.mirrorservice.org/sites/ftp.ibiblio.org/pub/docs/books/gutenberg/')).strip()
        text = ' '.join(' '.join(' '.join(text.split('\n')).split('\t')).split('\r'))
        text = ' '.join(text.split())
        text = clean_text(str(text))
    except:
        try:
            page = requests.get(row['Link'])
            soup = BeautifulSoup(page.content, 'html.parser')
            text_link = 'http://www.gutenberg.org' + soup.find_all("a", string="Plain Text UTF-8")[0]['href']
            http_response_object = urlopen(text_link)

            text = strip_headers(str(http_response_object.read()))
            text = ' '.join(' '.join(' '.join(text.split('\n')).split('\t')).split('\r'))
            text = ' '.join(text.split())
            text = clean_text(str(text))
        except:
            print("Couldn't acquire text for " + row['Title'] + ' with ID ' + str(book_id) + '. Link: ' + row['Link'])

    if data['Text'] == None:
        data['Text'] = [' '.join(text.split(' '))]
    else:
        try:
            data['Text'].append(' '.join(text.split(' ')))
        except:
            data['Text'].append(None)
            print("Couldn't save data for " + row['Title'] + ' with ID ' + str(book_id) + '. Link: ' + row['Link'])

df_data = pd.DataFrame(data, columns = ['Title', 'Author', 'Link', 'ID', 'Bookshelf', 'Text'])
with open('gutenberg_data.pkl','wb') as f:
   pickle.dump(df_data, f)

Couldn't acquire text for Solid Geometry with Problems and Applications (Revised edition) with ID 29807. Link: http://www.gutenberg.org/ebooks/29807
Couldn't save data for Solid Geometry with Problems and Applications (Revised edition) with ID 29807. Link: http://www.gutenberg.org/ebooks/29807
Couldn't acquire text for A Primer of Quaternions with ID 9934. Link: http://www.gutenberg.org/ebooks/9934
Couldn't save data for A Primer of Quaternions with ID 9934. Link: http://www.gutenberg.org/ebooks/9934
Couldn't acquire text for The History of England, from the Accession of James II, Volume 1, Chapter 04 with ID 20276. Link: http://www.gutenberg.org/ebooks/20276
Couldn't save data for The History of England, from the Accession of James II, Volume 1, Chapter 04 with ID 20276. Link: http://www.gutenberg.org/ebooks/20276
Couldn't acquire text for Pride and Prejudice with ID 20686. Link: http://www.gutenberg.org/ebooks/20686
Couldn't save data for Pride and Prejudice with ID 20686. Link: http